In [1]:
import networkx as nx
import matplotlib.pyplot as plt

print(f"NetworkX version: {nx.__version__}")

NetworkX version: 3.7


In this notebook, we focus on creating node level embeddings on graph  using path and cycle counting

# Path and Cycle Counting in Graphs ($P_l$ and $C_l$)

## 1. Definitions and Transition to the Node Level

For a simple undirected graph with $n$ vertices, we define the following matrices:

* $A \in \{0,1\}^{n \times n}$: Adjacency matrix (zero diagonal).
* $I \in \{0,1\}^{n \times n}$: Identity matrix.
* $J \in \{0,1\}^{n \times n}$: Matrix filled with $1$s off-diagonal and $0$s on the diagonal ($J = \mathbf{1}\mathbf{1}^\top - I$).

### Counting Matrices (Edge / Vertex-Pair Level)

* **Simple path matrix** $P_l$: $(P_l)_{i,j}$ counts the number of **simple paths** (without repeated vertices) of length $l$ connecting vertex $i$ to vertex $j$.
* **Simple cycle matrix** $C_l$ ($l \ge 3$): $(C_l)_{i,j}$ counts the number of **simple cycles** of length $l$ passing through the edge $(i,j)$.

### Fundamental Relationship

A simple cycle of length $l+1$ passing through $(i,j)$ is formed by a simple path of length $l$ between $i$ and $j$ closed by the direct edge $(i,j)$:

$$
C_{l+1} = A \odot P_l
$$

*(where $\odot$ denotes the Hadamard product, or element-wise multiplication).*

### Feature Extraction at the Node Level

* **Number of simple paths of length $l$ starting from node $i$:**

  $$
  \text{paths}_l(i) = \sum_{j=1}^n (P_l)_{i,j}
  $$

* **Number of simple cycles of length $l$ containing node $i$:**

  $$
  \text{cycles}_l(i) = \frac{1}{2} \sum_{j=1}^n (C_l)_{i,j}
  $$

  *(Factor of $\frac{1}{2}$ because each cycle passing through $i$ uses exactly two edges incident to $i$).*

## 2. Matrix Formulas ($P_l, C_l$ vs. Optimized Formulas $P_l^*, C_l^*$)

Powers $A^l$ count **walks**, which allow backtracking and internal cycles. To obtain $P_l$, these non-simple walks must be subtracted. The factored formulas ($P_l^*, C_l^*$), discovered via Grammar Reinforcement Learning (GRL), significantly reduce the number of matrix multiplications.

### Length 2 (2-Paths and Triangles)

* **Paths $P_2$:**

  $$
  P_2 = P_2^* = J \odot A^2
  $$

* **Triangles $C_3$:**

  $$
  C_3 = A \odot P_2 = A \odot A^2
  $$

### Length 3 (3-Paths and Squares)

* **Classical Formula (Voropaev & Perepechko):**

  $$
  P_3 = J \odot A^3 - (I \odot A^2)A - A(I \odot A^2) + A
  $$

  $$
  C_4 = A \odot P_3 = A \odot A^3 - A(I \odot A^2) - (I \odot A^2)A + A
  $$

* **Optimized Formula ($2\times$ faster):**

  $$
  P_3^* = J \odot \big(A(J \odot A^2)\big) - A \odot (AJ)
  $$

  $$
  C_4^* = A \odot P_3^*
  $$

### Length 4 (4-Paths and Pentagons)

* **Classical Formula:**

  $$
  P_4 = J \odot A^4 - J \odot \big(A(I \odot A^2)A\big) + 2(J \odot A^2) - (I \odot A^2)(J \odot A^2) - (J \odot A^2)(I \odot A^2) - A(I \odot A^3) - (I \odot A^3)A + 3A \odot A^2
  $$

* **Optimized Formula ($2.25\times$ faster):**

  $$
  P_4^* = J \odot \Big(A\big(J \odot (A(J \odot A^2))\big)\Big) - J \odot \big(A(A \odot (AJ))\big) - J \odot \big((A \odot (AJ))A\big) - A \odot \big((A \odot A^2)J\big) + 2A \odot A^2
  $$

  $$
  C_5^* = A \odot P_4^*
  $$

## 3. Theoretical Limit (3-WL)

The set of expressions generated from $\{A, I, J\}$ using matrix multiplication and the Hadamard product $\odot$ (grammar $\mathcal{G}_3$) has the exact same expressive power as the **3-Weisfeiler-Lehman (3-WL)** test:

* **Exact counting possible:** Paths up to length $l = 6$ ($P_6, P_6^*$) and cycles up to length $l = 7$ ($C_7, C_7^*$).
* **Fundamental limitation:** It is theoretically impossible using this matrix language to exact-count paths of length $l \ge 7$ or cycles of length $l \ge 8$.

# Node level